# ?? RSNA Knee Multimodal AI - Comprehensive Exploratory Data Analysis (EDA)

This notebook provides an in-depth exploratory data analysis of the **RSNA Knee MRI Multimodal AI** dataset.

### Problem Overview
- **Goal**: Predict probabilities for **12 clinically important knee abnormalities** evaluated via **Macro-Averaged AUC-ROC**.
- **12 Targets**:
  - `ACL`, `MCL`
  - `Medial Meniscus`, `Lateral Meniscus`
  - `Medial OA`, `Lateral OA`, `PF OA`
  - `Effusion`, `Synovitis`, `Baker's`
  - `Contusion`, `Fracture`
- **Data Modalities**:
  1. **Multi-sequence MRI Studies** (`train_series.csv`): Sagittal, Coronal, Axial series with varying contrast (`Fluid_Sensitive`, `Fat_Suppression`).
  2. **Radiology Reports** (`train.csv`): Free-text diagnostic reports written by radiologists in multiple languages.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})

DATA_DIR = Path('../data')
TARGET_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

print("Setup complete. Ready to load data.")

## 1. Data Ingestion & Cohort Overview

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
series_df = pd.read_csv(DATA_DIR / 'train_series.csv')

print(f"Total Studies in train.csv: {len(train_df):,}")
print(f"Total Series in train_series.csv: {len(series_df):,}")
print(f"Unique Studies with Series: {series_df['StudyInstanceUID'].nunique():,}")

labeled_df = train_df[train_df['ACL'].notna()].copy()
unlabeled_df = train_df[train_df['ACL'].isna()].copy()

print(f"\n? Studies with Gold-Standard Labels: {len(labeled_df):,} ({len(labeled_df)/len(train_df)*100:.2f}%)")
print(f"? Studies with Free-Text Reports for Weak Supervision: {len(unlabeled_df):,} ({len(unlabeled_df)/len(train_df)*100:.2f}%)")

## 2. Target Abnormality Prevalence (Gold Standard Cohort)

In [ ]:
gold_counts = labeled_df[TARGET_COLS].sum().astype(int)
gold_prev = (labeled_df[TARGET_COLS].mean() * 100).round(2)

label_stats = pd.DataFrame({
    'Positive_Cases': gold_counts,
    'Prevalence (%)': gold_prev
}).sort_values(by='Prevalence (%)', ascending=False)

display(label_stats)

plt.figure(figsize=(11, 6))
ax = sns.barplot(x=label_stats['Prevalence (%)'], y=label_stats.index, palette='crest')
plt.title(f"Gold Standard Target Prevalence (N={len(labeled_df)} Studies)", fontsize=14, fontweight='bold')
plt.xlabel("Prevalence (%)", fontsize=12)

for i, v in enumerate(label_stats['Prevalence (%)']):
    cnt = label_stats.loc[label_stats.index[i], 'Positive_Cases']
    ax.text(v + 0.8, i, f"{v:.1f}% ({cnt}/{len(labeled_df)})", va='center', fontweight='semibold')

plt.xlim(0, max(label_stats['Prevalence (%)']) + 12)
plt.show()

## 3. Abnormality Co-occurrence & Correlation Matrix

In [ ]:
corr = labeled_df[TARGET_COLS].corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0, square=True, cbar_kws={'shrink': 0.8})
plt.title("Target Co-occurrence Correlation Heatmap", fontsize=14, fontweight='bold')
plt.show()

## 4. Multi-Label Pathology Count per Patient

In [ ]:
labeled_df['num_pathologies'] = labeled_df[TARGET_COLS].sum(axis=1)

plt.figure(figsize=(8, 5))
path_counts = labeled_df['num_pathologies'].value_counts().sort_index()
ax = sns.barplot(x=path_counts.index, y=path_counts.values, palette='magma')
plt.title("Number of Co-occurring Pathologies per Study", fontsize=14, fontweight='bold')
plt.xlabel("Number of Pathologies (out of 12)", fontsize=12)
plt.ylabel("Number of Studies", fontsize=12)

for i, (idx, v) in enumerate(path_counts.items()):
    ax.text(i, v + 0.3, f"{v}", ha='center', fontweight='semibold')

plt.show()

## 5. MRI Acquisition & Series Metadata Analysis

In [ ]:
series_per_study = series_df.groupby('StudyInstanceUID').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Series count per study
s_counts = series_per_study.value_counts().sort_index()
sns.barplot(x=s_counts.index, y=s_counts.values, palette='viridis', ax=axes[0])
axes[0].set_title("MRI Series Available per Study (Total Cohort)", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Number of Series per Study")
axes[0].set_ylabel("Number of Studies")

# Anatomical planes
plane_counts = series_df['Anatomical_Plane'].value_counts()
sns.barplot(x=plane_counts.index, y=plane_counts.values, palette='Set2', ax=axes[1])
axes[1].set_title("Series Distribution by Anatomical Plane", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Anatomical Plane")
axes[1].set_ylabel("Count")

plt.show()

## 6. Radiology Report Text Analysis & Multi-lingual Samples

In [ ]:
train_df['report_words'] = train_df['Report'].fillna('').apply(lambda x: len(x.split()))

print("=== Report Summary Statistics ===")
print(f"Mean Word Count:   {train_df['report_words'].mean():.1f}")
print(f"Median Word Count: {train_df['report_words'].median():.1f}")
print(f"Max Word Count:    {train_df['report_words'].max():.1f}")

plt.figure(figsize=(9, 4.5))
sns.histplot(train_df['report_words'], bins=40, kde=True, color='teal')
plt.title("Radiology Report Word Count Distribution", fontsize=14, fontweight='bold')
plt.xlabel("Word Count per Report")
plt.ylabel("Frequency")
plt.show()

print("\n=== Sample Reports Across Languages ===")
for i, sample in enumerate(train_df['Report'].sample(3, random_state=42)):
    print(f"\n--- Sample {i+1} ---")
    print(sample[:300] + "...")

## 7. Leak-Free Multi-Label Stratified Validation Folds

In [ ]:
folds_df = pd.read_csv(DATA_DIR / 'train_folds.csv')

print("Folds distribution across all studies:")
display(folds_df['fold'].value_counts().sort_index().to_frame('Total Studies'))

print("\nGold-Standard Pathologies Distribution across Folds:")
gold_folds = folds_df[folds_df['ACL'].notna()]
display(gold_folds.groupby('fold')[TARGET_COLS].sum().astype(int))